# Global Market Regime Recognition Model ~ Data From October 2016 Onwards

Gaussian Mixture Model (GMM) to identify hidden market states across key regional macro indicators including equities, interest rates, currencies, commodities, and volatility.
Captures distinct regime shifts for the US, EU, and EM, SOAF markets using assets like SPX, 10-year yields, DXY, BCOM, and VIX, TOP40, CTZAR10Y Govt, SAVIT, etc.. to quantify patterns in risk, momentum, and macro behavior.

The results are presented as: 

1. An DataGrid showing mean returns and volatilities by regime and asset.
2. A line plot with regime-colored bands for each asset.
3. An area chart illustrating the posterior probabilities of each regime over time.


**Interactive features:**
- Use the dropdown to select a region (US, EU, or EM, SOAF) and the date pickers to specify an analysis period.

**Methodology:**

A Gaussian Mixture Model (GMM) is used to identify distinct market regimes based on changes in key macro indicators, including equities, interest rates, currencies, commodities, and volatility. The optimal number of regimes is selected using the Bayesian Information Criterion (BIC), with a cutoff applied when the improvement in BIC falls below 2%, balancing model fit with interpretability. Too many regimes can lead to overfitting and reduce the economic meaning of each state, making it harder to draw actionable insights. In general expect 2 or 3 regimes as output. This captures soft clustering —assigning probabilities rather than hard classifications—allowing for smoother transitions between regimes. Some regimes, may show 0 for return or standard deviation. This may be because the history is too short or the boundry for transition may be overwhelmed by another asset. For example: SOAF may have 3 regimes, but BCOMPR may have Daily Mean Return and Daily Volatility of zero becasue ZAR movement overwhelms this. 

To interpret the results, examine each regime’s mean returns and volatilities: a risk-off regime may feature falling equities, rising volatility, and rising yields (positive changes in bond basis points). For SOAF we use CTZAR10Y Govt as a price index, a <u>negative return is rising yields</u>. A risk-on regime may show broad asset strength, declining volatility, and falling yields (negative bps change- see corollary for SOAF yields). You can use these patterns to align market behavior with macro themes like tightening cycles, growth rebounds, or flight-to-safety conditions.

In [177]:
import os

# Limit OpenMP threads to 4 to prevent memory leaks in scikit-learn on Windows
os.environ["OMP_NUM_THREADS"] = "4"

import datetime

from dateutil.relativedelta import relativedelta

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture

import bqplot as bqp
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import ipywidgets as widgets
import ipydatagrid as ipd

import bql

In [178]:
# Instantiate connection to BQL
bq = bql.Service()

In [179]:
# Define regime mapping
region_mapping = {
    'US': {
        'SPX Index': 'pct_change',
        'USGG10YR Index': 'diff',
        'DXY Curncy': 'pct_change',
        'BCOM Index': 'pct_change',
        'VIX Index': 'diff'
    },
    'EU': {
        'SX5E Index': 'pct_change',
        'EUSA10 Curncy': 'diff',
        'EUR Curncy': 'pct_change',
        'BCOM Index': 'pct_change',
        #'VXMXEA Index': 'diff',
        #'VIX Index': 'diff',
        'V2X Index': 'diff'
    },
    'EM': {
        'MXEF Index': 'pct_change',
        'USGG10YR Index': 'diff',
        'BFXEMES Index': 'pct_change',
        'BCOM Index': 'pct_change',
        'VXEEM Index': 'diff'
    },
    'SOAF': {
        'TOP40 Index': 'pct_change',
        'CTZAR10Y Govt': 'pct_change',
        'BFXEMEES Index': 'pct_change',
        'USDZAR BGN Curncy': 'pct_change',
        'BCOMPR Index': 'pct_change',
        'SAVIT40 Index': 'diff'
        
        
        
    }
}

In [180]:
def get_asset_prices(universe, start_date, end_date):
    """
    Fetch asset price data.
    """      
    date_range = bq.func.range(start_date, end_date, frq='d')
    
    data_items = {
        'Price': bq.data.px_last(dates=date_range).dropna()
    } 
    request = bql.Request(
        universe, data_items, with_params={'mode': 'cached'}
    )
    response = bq.execute(request)
    
    data = response[0].df()
    data.reset_index(inplace=True)
    result = data.pivot(index='DATE', columns='ID', values='Price').ffill()
    
    return result

In [181]:
def compute_returns(df, asset_transforms):
    """
    Compute returns based on a method defined in mapping dictionary.
    (methods: 'pct_change' or 'diff')
    """
    return df.agg(asset_transforms).dropna()

In [182]:
def fit_model(data, n_components):
    """
    Fits a Gaussian Mixture Model (GMM) with n components.
    """
    model = GaussianMixture(
        n_components=n_components,
        covariance_type='diag',
        max_iter=1000,
        random_state=42
    )
    return model.fit(data)

In [183]:
def fit_gmm(data, n_components):
    """
    Fit a Gaussian Mixture Model (GMM) with n components and return the
    BIC score.
    """
    model = fit_model(data, n_components)
    return model.bic(data)

In [184]:
def predict_gmm_states(data, n_components, asset_returns):
    """
    Fit a GMM to the scaled return data and return regime state predictions.
    """
    model = fit_model(data, n_components)
    states = model.predict(data) + 1
    return pd.Series(states, index=asset_returns.index)

In [185]:
def plot_elbow(bics):
    """
    Create a Plotly figure to visualize BIC scores across different
    regime counts.
    """
    fig = go.FigureWidget(data=go.Scatter(
        x=list(range(1, 7)),
        y=bics,
        mode='lines+markers',
        marker=dict(color='blue', size=8),
        line=dict(color='blue')
    ))
    
    fig.update_layout(
        title='Elbow Plot for Gaussian Mixture Model',
        xaxis_title='Number of Regimes',
        yaxis_title='BIC',
        width=550,
        height=400,
        template='plotly_dark'
    )   
    # Axis settings: whole numbers on x-axis
    fig.update_xaxes(
        title='Number of Regimes',
        tickmode='linear',  # ensure regular spacing
        dtick=1,            # integer steps
        tick0=1,            # start at 1
        tickformat=',d'     # format as integers (no decimals)
    )
    return fig

In [186]:
min_improvement=0.02
def find_elbow_by_improvement(bics, min_improvement=min_improvement):
    """
    Find the smallest number of regimes where BIC improvement falls below 2%.
    Returns the optimal number of components based on diminishing BIC gains.
    """
    for i in range(1, len(bics)):
        delta = bics[i-1] - bics[i]
        if delta / abs(bics[i-1]) < min_improvement:
            return i  
    return len(bics)

In [187]:
def prepare_data(returns_data):
    """
    Scales returns for GMM input.
    """
    scaler = StandardScaler()
    return scaler.fit_transform(returns_data)

In [188]:
def create_empty_figure_widget(asset_returns):
    """
    Create an empty multi-row subplot figure for the asset list.
    """
    num_assets = len(asset_returns.columns)
    
    fig = go.FigureWidget(make_subplots(
        rows=num_assets,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.12
    ))

    fig.update_layout(
        title=dict(text='Asset Regimes', x=0.5),
        height=300 * num_assets,
        width=1100,
        margin=dict(t=80, b=40),
        template='plotly_dark'
    )
    # Format axes
    for i in range(1, num_assets + 1):
        fig.update_xaxes(showticklabels=True, tickangle=45, row=i, col=1)
        fig.update_yaxes(title_text="", row=i, col=1)
    return fig

In [189]:
def update_asset_subplots(fig, asset_prices, states, n_components):
    """
    Plot asset time series (filtered_assets) as individual subplots and 
    overlay regime segments from state_series as background bands. Each
    subplot corresponds to one asset, showing its price evolution alongside
    GMM-inferred regime changes over time.
    """
    
    regime_colors = {
        1: "#32CD32",   # Bright Green
        2: "#1E90FF",   # Bright Blue
        3: "#FF4C4C",   # Bright Red
        4: "#FFD700",   # Bright Yellow
        5: "#00E7FF",   # Light Blue
        6: "#FF9999",   # Light Red
    }
    
    states = pd.DataFrame({'regime': states}).reset_index()
    states = states[states['regime'].transform(lambda x: x.diff() != 0)]
    states['end_date'] = states['DATE'].shift(-1) 

    states.loc[states.index[-1], 'end_date'] = asset_prices.index[-1]
    
    all_shapes = []
    for row_idx, asset in enumerate(asset_prices.columns, start=1):
        # Plot time series line
        ts = asset_prices[asset].values 
        fig.add_trace(
            go.Scatter(
                x=asset_prices.index,
                y=ts,
                mode='lines',
                line=dict(color='white', width=2),
                showlegend=False,
                hovertemplate="Date: %{x}<br>Value: %{y}<extra></extra>"
            ), 
            row=row_idx,
            col=1
        )

        # update subchart title 
        fig.update_yaxes(title_text=asset, row=row_idx, col=1)
        
        # Set y-axis range
        y_min = float(np.nanmin(ts)) * 0.995
        y_max = float(np.nanmax(ts)) * 1.005
        fig.update_yaxes(range=[y_min, y_max], row=row_idx, col=1)
        # Axis refs
        xref = f"x{row_idx}" if row_idx > 1 else "x"
        yref = f"y{row_idx} domain" if row_idx > 1 else "y domain"
        # Add background regime bands
        for idx, row in states.iterrows():
            color = regime_colors.get(row['regime'], '#999999')
            all_shapes.append(dict(
                type="rect",
                x0=row['DATE'],
                x1=row['end_date'] + pd.Timedelta(days=1),
                y0=0,
                y1=1,
                xref=xref,
                yref=yref,
                fillcolor=color,
                opacity=1.0,
                line_width=0,
                layer="below"
            ))
        
        # Only show regime labels for regimes 1 to n_components
        regime_text = ""
        for i in range(1, n_components + 1):
            color = regime_colors.get(i, "#999999")
            regime_text += (
                f"<span style='color:{color}'><b>Regime {i}</b></span>"
                "&nbsp;&nbsp;&nbsp;&nbsp;"
            )
            
        # Position labels below each subplot
        axis_key = f'yaxis{"" if row_idx == 1 else row_idx}'
        y_domain = fig.layout[axis_key].domain
        y_paper = y_domain[0] - 0.055
        fig.add_annotation(
            xref='paper',
            yref='paper',
            x=0.5,
            y=y_paper,
            showarrow=False,
            align="center",
            text=regime_text,
            font=dict(size=12),
            xanchor="center"
        )
    # Apply shapes to layout
    fig.layout.shapes = all_shapes
    
    # Final layout settings
    fig.update_layout(
        height=300 * len(asset_prices.columns),
        width=1100,
        margin=dict(t=100, b=120),
        title=dict(text='Asset Regimes', x=0.5),
        template='plotly_dark'
    )
    fig.update_xaxes(tickangle=45, row=len(asset_prices.columns), col=1)

In [190]:
def calculate_posterior_probabilities(
    scaled_returns, 
    asset_returns, 
    n_components
):
    """
    Fit a GMM to the scaled return data and compute posterior probabilities
    for each regime over time. Returns both the full regime probability
    matrix and cumulative probabilities for use in stacked visualizations.
    """
    gmm = fit_model(scaled_returns, n_components)
    posteriors = gmm.predict_proba(scaled_returns)

    posterior_df = pd.DataFrame(
        posteriors,
        index=asset_returns.index,
        columns=[f'Prob_regime_{i+1}' for i in range(n_components)]
    )
    
    # Cumulative probabilities for stacked area plot
    cumulative_probs = posterior_df.cumsum(axis=1)

    return posterior_df, cumulative_probs

In [191]:
def create_posterior_chart(cumulative_probs):
    """
    Create a stacked area chart showing posterior probabilities over time for
    any number of regimes.
    """
    
    regime_colors = {
        1: "#32CD32",   # Bright Green
        2: "#1E90FF",   # Bright Blue
        3: "#FF4C4C",   # Bright Red
        4: "#FFD700",   # Bright Yellow
        5: "#00E7FF",   # Light Blue
        6: "#FF9999",   # Light Red
    }
    
    fig = go.FigureWidget()
    
    # Extract regime numbers from column names
    regime_numbers = []
    for col in cumulative_probs.columns:
        # Extract the number at the end of each column name
        number = int(col.split('_')[-1])
        regime_numbers.append(number)
    
    for i, col in enumerate(cumulative_probs.columns):
        regime_num = regime_numbers[i] 
        
        if i == 0:
            lower = np.zeros(len(cumulative_probs.index))
        else:
            lower = cumulative_probs.iloc[:, i - 1]
    
        # Lower bound trace (invisible)
        fig.add_trace(go.Scatter(
            x=cumulative_probs.index,
            y=lower,
            mode='lines',
            line=dict(width=0),
            showlegend=False
        ))
    
        fig.add_trace(go.Scatter(
            x=cumulative_probs.index,
            y=cumulative_probs.iloc[:, i],
            mode='lines',
            line=dict(width=0),
            fill='tonexty',
            fillcolor=regime_colors.get(regime_num, "#999999"),
            showlegend=False,  
            opacity=0.5
        ))
    
    fig.update_layout(
        title={
            'text': 'Regime Posterior Probabilities',
            'x': 0.5,  
            'xanchor': 'center',
            'yanchor': 'top' 
        },
        xaxis_title=None,
        yaxis_title='Probability',
        yaxis=dict(range=[0, 1]),
        width=1100,
        height=400,
        template='plotly_dark',
        margin=dict(t=50, b=50),
        showlegend=False 
    )
    return fig

In [192]:

def calc_regime_stats(df, agg, selected_assets):
    stats = df.groupby('Regime').agg(agg)
    stats['Freq (%)'] = df['Regime'].value_counts(normalize=True)
    stats = rename_asset_headings(stats, selected_assets)
    stats = apply_mulitplier(stats)
    return stats

def get_transform_symbol(asset, selected_assets):
    """
    Decide the unit suffix for an asset column heading.
    - VIX-like assets return '(pts)'.
    - Otherwise, look up transform in selected_assets: 'pct_change' -> '(%)', 'diff' -> '(bps)'.
    - If not found, default to no suffix.
    """
    vix_like = {'VIX', 'VXEEM', 'SAVIT40', "V2X", "VXMXEA"}  # extend this list as needed
    transform_symbols = {
        'pct_change': '(%)',
        'diff': '(bps)',
    }

    if asset in vix_like:
        return '(pts)'

    # Safely handle assets not present in selected_assets
    transform = selected_assets.get(asset)
    if transform is None:
        # Fallback: unknown transform -> no suffix
        return ''

    return transform_symbols.get(transform, '')

def apply_mulitplier(df):
    """
    Multiply columns that are percentage or basis points by 100.
    Point columns '(pts)' are left unchanged.
    """
    for col in df.columns:
        if '(%)' in col or '(bps)' in col:
            df[col] = df[col] * 100
    return df

def rename_asset_headings(df, selected_assets):
    """
    Rename columns to '<ASSET> <unit>' where unit is decided by get_transform_symbol.
    For multi-word asset names, the first token is preserved as in original code (asset.split(' ')[0]).
    """
    rename = {}
    for asset in selected_assets.keys() | set(df.columns):
        if asset in df.columns:
            symbol = get_transform_symbol(asset, selected_assets)
            # Keep only the first token of the asset name, as per your original logic
            new_name = asset.split(' ')[0] + (' ' + symbol if symbol else '')
            rename[asset] = new_name
    return df.rename(columns=rename)


In [193]:
def build_combined_renderers(df):
    """
    Create renderers for color coding both return and volatility columns in a
    combined datagrid.
    """
    renderers = {}
    
    returns_renderer = ipd.TextRenderer(
        background_color=ipd.VegaExpr("cell.value > 0 ? 'green' : 'red'"),
        horizontal_alignment='center',
        format='.4f'
    )
    
    vol_renderer = ipd.TextRenderer(
        text_color='rgb(20,20,20)',
        horizontal_alignment='center',
        format="0.3f",
        background_color=bqp.ColorScale(
            min=df['Daily Volatility'].max(),  
            max=df['Daily Volatility'].min(),
            mid=3.0,
            scheme='RdYlGn'
        )
    )

    regime_renderer = ipd.TextRenderer(
        horizontal_alignment='center',
        vertical_alignment='center'
    )
    
    # Assign return renderers
    renderers['Daily Mean Return'] = returns_renderer
    renderers['Daily Volatility'] = vol_renderer
    renderers['Regime'] = regime_renderer
    return renderers

In [194]:
def build_combined_stats_grid(returns_df, vol_df, selected_assets):
    """
    Creates a combined grid with Asset, Regime, Daily Mean Return, and
    Daily Volatility.
    """
    # Create a new dataframe with the combined data
    combined_data = []
    
    for asset in selected_assets.keys():
        asset_name = asset.split(' ')[0]  
        
        # Get the column name for this asset in returns and vol dataframes
        return_col = [col for col in returns_df.columns if asset_name in col][0]
        vol_col = [col for col in vol_df.columns if asset_name in col][0]
        
        # For each regime, get return and volatility
        for regime in returns_df.index:
            combined_data.append({
                'Asset': return_col,
                'Regime': regime,
                'Daily Mean Return': returns_df.loc[regime, return_col],
                'Daily Volatility': vol_df.loc[regime, vol_col]  #
            })
    
    combined_df = pd.DataFrame(combined_data)

    # Create renderers for the combined grid
    renderers = build_combined_renderers(combined_df)
    
    # Header renderer
    header_renderer = ipd.TextRenderer(
        vertical_alignment='top',
        background_color='rgb(50,50,50)',
        horizontal_alignment='center',
    )
      
    # Create the grid
    grid = ipd.DataGrid(
        dataframe=combined_df.set_index('Asset'),
        header_renderer=header_renderer,
        renderers=renderers,
        base_column_size=105,
        base_row_header_size=100,
        base_column_header_size=40,
        layout={'width': '450px', 'height': '300px'}  
        
    )
    return grid

In [195]:
def update_status(message):
    """
    Update the status message widget.
    """
    status_output.value = f"<span style='color: white'>{message}</span>"

def run(event=None):
    """
    Main function to run the analysis and update the UI.
    """
    stats_title.layout.visibility = 'hidden'
    current_regime.layout.visibility = 'hidden'

    exception_box.children = []
    elbow_plot_box.children = []
    regime_plot_box.children = []
    posterio_plot_box.children = []
    grid_box1.children = []
    spinner.layout.visibility = "visible"

    try: 
        # Get selected regime
        selected_assets = region_mapping[regime_dropdown.value]
        update_status(f"Fetching data for {len(selected_assets)} assets")

        # Step 1: Get asset prices and compute returns
        asset_prices = get_asset_prices(
            selected_assets,
            start_date.value,
            end_date.value)
        
        update_status("Asset prices retrieved, computing returns")
        asset_returns = compute_returns(asset_prices, selected_assets)

        # Step 2: Prepare data
        scaled_returns_array = prepare_data(asset_returns)

        # Step 3: Calculate GMM for elbow plot
        update_status("Calculating optimal number of regimes")
        bics = [fit_gmm(scaled_returns_array, n) for n in range(1, 4)]#
        elbow_plot = plot_elbow(bics)
        n_components = find_elbow_by_improvement(bics, min_improvement=min_improvement)
        update_status("number of regimes:" f'{n_components}')

        states = predict_gmm_states(
            scaled_returns_array, 
            n_components, 
            asset_returns
        )

        regime = states.iloc[-1]
        current_regime.value = f"<h5>Current Regime: {regime}</h5>"

        # Step 4: Create regime plot
        update_status("Creating regime visualizations")
        regime_fig = create_empty_figure_widget(asset_returns)
        
        update_asset_subplots(
            regime_fig,
            asset_prices,
            states, 
            n_components
        )

        # Step 5: Create posterior probability plot
        update_status("Calculating regime probabilities")

        posterior_df, cumulative_probs = calculate_posterior_probabilities(
            scaled_returns_array,
            asset_returns,
            n_components
        )
        posterior_plot = create_posterior_chart(cumulative_probs)

        # Step 6: Calculate regime returns and volatilities
        update_status("Calculating regime statistics")
        asset_returns['Regime'] = states

        returns_summary = calc_regime_stats(
            asset_returns, 'mean', selected_assets
        )
        vol_summary = calc_regime_stats(
            asset_returns, 'std', selected_assets
        )

        # Create combined grid instead of separate ones
        combined_grid = build_combined_stats_grid(
            returns_summary, 
            vol_summary, 
            selected_assets
        )

        # Step 7: Update UI
        update_status("Updating plots")
        elbow_plot_box.children = [elbow_plot]
        regime_plot_box.children = [regime_fig]
        posterio_plot_box.children = [posterior_plot]
        grid_box1.children = [combined_grid]

        update_status("Analysis complete")
        update_status("")

    except Exception as e:
        exception_box.children = [widgets.Label(f'{e}')]
        update_status("Analysis failed. See error details above.")

    finally:
        spinner.layout.visibility = 'hidden'
        stats_title.layout.visibility = 'visible'
        current_regime.layout.visibility = 'visible'

In [196]:
# UI Components
regime_dropdown = widgets.Dropdown(
    options=[
        ('US Regime', 'US'), 
        ('European Regime', 'EU'), 
        ('EM Regime', 'EM'),
        ('Soth Africa Regime','SOAF')
    ],
    value='US',
    description='Select Regime:',
    layout={'width': '250px'},
    style={'description_width': 'initial'}
)
three_years_ago = datetime.date.today() - relativedelta(years=3)
start_date = widgets.DatePicker(
    description='Start Date',
    value=three_years_ago,
    layout={'width': '190px'},
    style={'description_width': 'initial'}
)
end_date = widgets.DatePicker(
    description='End Date',
    value=datetime.date.today(),
    max=datetime.date.today(),
    layout={'width': '190px'},
    style={'description_width': 'initial'}
)
spinner = widgets.HTML(
    '<i class="fa fa-spinner fa-spin" style="font-size: 18px"></i>',
    layout={
        'visibility': 'hidden', 
        'margin': '3px 0 0 10px', 
        'width': '40px'
    }
)
status_output = widgets.HTML(
    value="<span style='color: white'>Ready</span>",
    layout={'width': '100%', 'margin': '5px 0'}
)
go_button = widgets.Button(
    description="Go",
    style={"description_width": "initial"},
    layout={"width": "60px", "margin": "3px 0 10px 0"},
    button_style="success",
)
# Register the button to the run function
go_button.on_click(run)
exception_box = widgets.VBox(
    layout={'margin': '10px 0', 'max-height': '200px', 'overflow': 'auto'}
)
elbow_plot_box = widgets.VBox()
regime_plot_box = widgets.VBox()
posterio_plot_box = widgets.VBox()
grid_box1 = widgets.VBox()

app_title = widgets.HTML(
    value="<h3>Global Market Regime Recognition Model</h3>"
)
stats_title = widgets.HTML(
    value="<h5>Asset Performance by Regime</h5>",
    layout=widgets.Layout(display='hidden', margin='10px 0 0 0')
)
current_regime = widgets.HTML(
    value=" ",
    layout=widgets.Layout(display='hidden', margin='10px 0 0 0')
)

ui_display = widgets.VBox([
    app_title,
    regime_dropdown,
    widgets.HBox([start_date, end_date]),
    widgets.HBox([go_button, spinner]),
    status_output,
    exception_box,
    widgets.HBox([
        elbow_plot_box,
        widgets.VBox([
            #empty_box,
            stats_title,
            grid_box1,
            current_regime
        ])
    ]),
    regime_plot_box,
    posterio_plot_box,
])

In [197]:
# Run on startup
run()

ui_display

In [198]:
#!jupyter nbconvert --to script RegimeMachine.ipynb

[NbConvertApp] Converting notebook RegimeMachine.ipynb to script
[NbConvertApp] Writing 25172 bytes to RegimeMachine.py
